<a href="https://colab.research.google.com/github/Samriddhacoderho/REAL_TIME_PEOPLE_COUNTING_SYSTEM/blob/main/people_counting_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO("yolov10n.pt")


results = model.predict(source="/content/8853A74A-E2DE-4914-BE7A-BE668453C861.JPG", save=True)
cls_names=model.names #sab classes stored in dict format
print(cls_names)
for result in results:
    img = result.plot()
    resized_img = cv2.resize(img, (800, 600))
    # cv2_imshow(resized_img)
    boxes=result.boxes
    print(len(boxes)) #3 aaucha output since there are 3 detected objects
    print(boxes.cls)
    for box in boxes:
      print(cls_names[int(box.cls)])


image 1/1 /content/8853A74A-E2DE-4914-BE7A-BE668453C861.JPG: 640x384 1 person, 1 chair, 1 laptop, 8.6ms
Speed: 2.1ms preprocess, 8.6ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)
Results saved to /content/runs/detect/predict-2
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: '

In [ ]:
!rm -rf '/content/runs/detect/predict-2'

In [ ]:
#Some basics of video capture in open cv before moving forward
video_path='/content/IMG_0628.mp4'
cap=cv2.VideoCapture(video_path) #cap chai euta video reader jastai ho, it acts like a pointer that opens the video file, and lets you read frames one by one.
#Video-->sequence of images(frames)

success,frame=cap.read()
print(success)
print(frame.shape) #so if frame.shape=(1920,1080,3) it means 1920 is hieght of that frame(1920 pixels), 1080 is width of that frame(1080 pixels), and 3 channels
print(frame[0,0]) #this gives BGR value for first pixel

#Video properties
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))
tot_frames_in_video=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Width of video is {width}\nheight is: {height}\nfps is:{fps}\ntotal frames in video is:{tot_frames_in_video}")

True
(1080, 1920, 3)
[66 80 93]
Width of video is 1920
height is: 1080
fps is:60
total frames in video is:1061


In [ ]:
out = cv2.VideoWriter(
    'output_count.mp4',
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

line_y = width // 2  # Draw line in the middle

In [ ]:
track_history = {}
counted_ids = set()
count = 0
offset=5

while (cap.isOpened()):
  success, frame = cap.read()
  if not success:
    break

  results = model.track(frame, persist=True, classes=[0], tracker="bytetrack.yaml")

  if(results[0].boxes.id is not None):
    boxes = results[0].boxes.xyxy.int().cpu().tolist()
    track_ids=results[0].boxes.id.int().tolist()

    for box,id in zip(boxes,track_ids):
      x1,y1,x2,y2=box
      cx=(x1+x2)//2
      cy=(y1+y2)//2
      cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
      cv2.putText(frame, f'ID:{str(id)}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
      if(id in track_history):
        prev_cx=track_history[id]
        if(prev_cx>(line_y) and cx<=(line_y)):
          if(id not in counted_ids):
            count+=1
            counted_ids.add(id)
      track_history[id]=cx

  cv2.line(frame, (line_y, 0), (line_y, height), (255, 0, 0), 3)
  cv2.putText(frame, f"Total People: {count}", (50, 50),cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)

  out.write(frame)

cap.release()
out.release()
print(f"Finished! Total count: {count}. Video saved as output_count.mp4")


0: 384x640 1 person, 9.1ms
Speed: 2.2ms preprocess, 9.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 12.2ms
Speed: 3.1ms preprocess, 12.2ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8.9ms
Speed: 3.0ms preprocess, 8.9ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 11.0ms
Speed: 2.8ms preprocess, 11.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8.8ms
Speed: 3.0ms preprocess, 8.8ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 12.7ms
Speed: 5.2ms preprocess, 12.7ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 18.3ms
Speed: 4.7ms preprocess, 18.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 11.8ms
Speed: 3.0ms preprocess, 11.8ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640